In [1]:
import json
import os
import sys
import pandas as pd
from pprint import pprint
from tqdm import tqdm
from langchain import PromptTemplate, FewShotPromptTemplate

sys.path.insert(0, '../src/')
from prompts import DEPRESSION_FEWSHOT_LANGCHAIN

tqdm.pandas()

In [2]:
DEPRESSION_FEWSHOT_LANGCHAIN

{'few_shot_prefix': "\nBelow are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.\nFormat your response as a JSON object /{'depression':''/} with values either 'yes' or 'no'.\n",
 'prompt_template': '\nPost: {post}\nAssesement: {label}\n',
 'few_shot_suffix': '\nBased on the above, assess the content of the following post:\nPost: {post}\nAssessment:\n'}

In [38]:
test_path = '../data/test/full_test.csv'
corpus_path = '../data/silver_data/silver_labels_gpt.csv'
mapping_path_depression = "../data/mappings/semantic-similarity-depression.json"
mapping_path_anxiety = "../data/mappings/semantic-similarity-anxiety.json"
mapping_path_comorbid = "../data/mappings/semantic-similarity-comorbid.json"
mapping_path_normal = "../data/mappings/semantic-similarity-normal.json"


df_test = pd.read_csv(test_path).reset_index()
df_corpus = pd.read_csv(corpus_path).reset_index()
mapping_depression = json.load(open(mapping_path_depression))
mapping_anxiety = json.load(open(mapping_path_anxiety))
mapping_comorbid = json.load(open(mapping_path_comorbid))
mapping_normal = json.load(open(mapping_path_normal))


print('-'*50)
print(f"Test set size: {df_test.shape[0]}")
print(f"Corpus (Silver Labels) size: {df_corpus.shape[0]}")
print(f"Total mapping (Depression): {len(mapping_depression)}")
print(f"Total mapping (Anxiety): {len(mapping_anxiety)}")
print(f"Total mapping (Comorbid): {len(mapping_comorbid)}")
print(f"Total mapping (Normal): {len(mapping_normal)}")
print('-'*50)


--------------------------------------------------
Test set size: 2872
Corpus (Silver Labels) size: 7667
Total mapping (Depression): 2872
Total mapping (Anxiety): 2872
Total mapping (Comorbid): 2872
Total mapping (Normal): 2872
--------------------------------------------------


In [39]:
def merge_dicts(dict1, dict2):
    merged_dict = {}
    for i in dict1.keys():
        merged_dict[i] = dict1[i]
    for i in dict2.keys():
        merged_dict[i] = dict2[i]
    return merged_dict

df_corpus['depression_label'] = df_corpus['depression_label'].apply(lambda label: {"depression": "yes"} if label == 1 else {"depression": "no"})
df_corpus['anxiety_label'] = df_corpus['anxiety_label'].apply(lambda label: {"anxiety": "yes"} if label == 1 else {"anxiety": "no"})
df_corpus['comorbidity_label'] = df_corpus.apply(lambda row: merge_dicts(row['depression_label'],row['anxiety_label']), axis=1)
df_corpus['comorbidity_label'].value_counts()

{'depression': 'no', 'anxiety': 'no'}      3349
{'depression': 'yes', 'anxiety': 'no'}     2705
{'depression': 'no', 'anxiety': 'yes'}     1048
{'depression': 'yes', 'anxiety': 'yes'}     565
Name: comorbidity_label, dtype: int64

In [40]:
df_corpus.head(1)

,level_0,index,text,subreddit,author,system_description,prompt,gpt_prediction,Mental Health Disorder,Name of Mental Health Disorder,DSM5 Rationale,silver_label,id,depression_label,anxiety_label,multilabel_clf_label,multiclass_clf_label,comorbidity_label
0,0,11,"i woke up very early, 2 am. i just came out of...",ForeverAlone,SixViking,You are a Psychology professor working in the ...,"Reddit Post: ""i woke up very early, 2 am. i ju...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Major Depressive Disorder (MDD),The language used in the post indicates severa...,Depression,4JEVyZ,{'depression': 'yes'},{'anxiety': 'no'},"[1, 0]",Depression,"{'depression': 'yes', 'anxiety': 'no'}"


In [64]:
DEPRESSION_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical depression as defined in the DSM-5.
Format your response as a JSON object {'depression': ''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

ANXIETY_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical anxiety as defined in the DSM-5.
Format your response as a JSON object {'anxiety':''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

COMORBIDITY_FEWSHOT_LANGCHAIN = {
"few_shot_prefix": '''
Below are posts and their respective assessments based on the criteria for clinical depression and clinical anxiety respectively as defined in the DSM-5.
Format your response as a JSON object {'depression': '', 'anxiety': ''} with values either 'yes' or 'no'.
'''
,
"prompt_template": lambda post, label: f'''
Post: {post}
Assesement: {label}
'''
,
"few_shot_suffix": lambda post: f'''
Based on the above, assess the content of the following post:
Post: {post}
Assessment:
'''
}

In [67]:
# Generate Few Shot prompts for each data point in test set.
topk = 4 # Provide only even number of examples, 2, 4, 6, 8, ..
text_col = 'text'
id_col = 'id'
task_type = 'comorbidity' #(depression | anxiety | comorbidity)

prompts = []
for i in tqdm(range(df_test.shape[0]), desc=f"Generating few shot prompts"):
    few_shot_examples = []
    input_post = df_test.iloc[i][text_col]
    
    if task_type == 'depression':        
        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_n
 
        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])
        
        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['depression_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = DEPRESSION_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = DEPRESSION_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = DEPRESSION_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
        
    
    elif task_type == 'anxiety':
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/2)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/2)]
        exemplar_ids = exemplar_ids_a + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['anxiety_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = ANXIETY_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = ANXIETY_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = ANXIETY_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
        
        
    elif task_type == 'comorbidity':
        if topk < 4: 
            topk = 4

        exemplars_d = mapping_depression[df_test.iloc[i][id_col]]
        exemplar_ids_d = [i[0] for i in exemplars_d][:int(topk/4)]
        exemplars_a = mapping_anxiety[df_test.iloc[i][id_col]]
        exemplar_ids_a = [i[0] for i in exemplars_a][:int(topk/4)]
        exemplars_c = mapping_comorbid[df_test.iloc[i][id_col]]
        exemplar_ids_c = [i[0] for i in exemplars_c][:int(topk/4)]
        exemplars_n = mapping_normal[df_test.iloc[i][id_col]]
        exemplar_ids_n = [i[0] for i in exemplars_n][:int(topk/4)]
        exemplar_ids = exemplar_ids_d + exemplar_ids_a + exemplar_ids_c + exemplar_ids_n

        exemplar_df = df_corpus[df_corpus[id_col].isin(exemplar_ids)]
        exemplar_df = exemplar_df.sample(exemplar_df.shape[0])

        for j, row in exemplar_df.iterrows():
            exemplar_post = row['text']
            exemplar_label = row['comorbidity_label']
            
            few_shot_examples.append(
                {
                    'post': exemplar_post,
                    'label': exemplar_label,
                }
            )
            
        prompt_template = COMORBIDITY_FEWSHOT_LANGCHAIN['prompt_template']
        few_shot_prefix = COMORBIDITY_FEWSHOT_LANGCHAIN['few_shot_prefix']
        few_shot_suffix = COMORBIDITY_FEWSHOT_LANGCHAIN['few_shot_suffix'](input_post)
               
        
    few_shot_examples = ''.join(prompt_template(x['post'], x['label']) for x in few_shot_examples)
    few_shot_prompt = ''.join([few_shot_prefix, few_shot_examples, few_shot_suffix])        
    prompts.append(few_shot_prompt)


df_test[f'few_shot_prompt_{task_type}'] = prompts
df_test[f'few_shot_prompt_{task_type}']

Generating few shot prompts:   0%|          | 0/2872 [00:00<?, ?it/s]Generating few shot prompts: 100%|██████████| 2872/2872 [00:04<00:00, 613.91it/s]


0       \nBelow are posts and their respective assessm...
1       \nBelow are posts and their respective assessm...
2       \nBelow are posts and their respective assessm...
3       \nBelow are posts and their respective assessm...
4       \nBelow are posts and their respective assessm...
                              ...                        
2867    \nBelow are posts and their respective assessm...
2868    \nBelow are posts and their respective assessm...
2869    \nBelow are posts and their respective assessm...
2870    \nBelow are posts and their respective assessm...
2871    \nBelow are posts and their respective assessm...
Name: few_shot_prompt_comorbidity, Length: 2872, dtype: object

### Post-Process GPT4 predictions

In [1]:
import ast
import numpy as np

def postprocess_label(pred_label, label_type):
    if label_type == 'depression':
        if isinstance(pred_label, str):
            pred_label = ast.literal_eval(pred_label)
        return 1 if pred_label == {"depression": "yes"} else 0
    
    elif label_type == 'anxiety':
        if isinstance(pred_label, str):
            pred_label = ast.literal_eval(pred_label)
        return 1 if pred_label == {"anxiety": "yes"} else 0
    
    elif label_type == 'comorbidity':
        if isinstance(pred_label, str):
            pred_label = ast.literal_eval(pred_label)
        
        if pred_label == {"depression": "yes", "anxiety": "yes"}:
            return json.dumps([1,1])
        elif pred_label == {"depression": "yes", "anxiety": "no"}:
            return json.dumps([1,0])
        elif pred_label == {"depression": "no", "anxiety": "yes"}:
            return json.dumps([0,1])
        elif pred_label == {"depression": "no", "anxiety": "no"}:
            return json.dumps([0,0])
        else:
            raise Exception(
                f"Encountered invalid label :{pred_label}"
            )

In [9]:
filename = '/home/ameyh/mental-health-comorbitidy-classification/results/few_shot/few_shot_comorbidity_gpt-3.5-turbo_num_examples_ss_4_seed_0.csv'
df_pred = pd.read_csv(filename)
print(df_pred.shape)
df_pred.head(1)

(2872, 13)


,index,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label,few_shot_prompt_comorbidity,exemplar_labels_comorbidity,results_comorbidity_gpt-3.5-turbo
0,0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0,\nBelow are posts and their respective assessm...,"[{'depression': 'yes', 'anxiety': 'no'}, {'dep...","{'depression': 'yes', 'anxiety': 'no'}"


In [10]:
df_test = pd.read_csv('/home/ameyh/mental-health-comorbitidy-classification/data/test/full_test.csv')

assert df_pred.shape[0] == df_test.shape[0]
df_pred['multilabel_clf_label'] = df_test['multilabel_clf_label']
df_test['multilabel_clf_label'].value_counts()

[0, 0]    1201
[1, 0]     970
[1, 1]     609
[0, 1]      92
Name: multilabel_clf_label, dtype: int64

In [11]:
df_pred['results_comorbidity_gpt-3.5-turbo'].value_counts()

{'depression': 'yes', 'anxiety': 'no'}     1562
{'depression': 'yes', 'anxiety': 'yes'}     882
{'depression': 'no', 'anxiety': 'yes'}      283
{'depression': 'no', 'anxiety': 'no'}       145
Name: results_comorbidity_gpt-3.5-turbo, dtype: int64

In [12]:
df_pred['results_comorbidity_gpt-3.5-turbo'].isna().sum()

0

In [13]:
df_pred['predicted_comorbidity_label'] = df_pred['results_comorbidity_gpt-3.5-turbo'].apply(lambda label: postprocess_label(label, label_type='comorbidity'))
df_pred['predicted_comorbidity_label'].value_counts()

[1, 0]    1562
[1, 1]     882
[0, 1]     283
[0, 0]     145
Name: predicted_comorbidity_label, dtype: int64

In [8]:
df_pred.to_csv('/home/ameyh/depository/mental-health-comorbitidy-classification/project/results/few_shot_comorbidity_gpt-3.5-turbo_num_examples_ss_4_seed_0.csv', index=False)

In [40]:
df = pd.read_csv('/home/ameyh/mental-health-comorbitidy-classification/predictions/depression_preds.csv')
df.columns

Index(['zero_shot_depression_naive_gpt-3.5-turbo_seed_0',
       'zero_shot_depression_naive_gpt-4_seed_0',
       'zero_shot_depression_mards_gpt-3.5-turbo_seed_0_t=35',
       'zero_shot_depression_mards_gpt-3.5-turbo_seed_0_t=20',
       'zero_shot_depression_mards_gpt-4_seed_0_t=35',
       'zero_shot_depression_mards_gpt-4_seed_0_t=20',
       'zero_shot_depression_phq9_gpt-3.5-turbo_seed_0',
       'zero_shot_depression_phq9_gpt-4_seed_0', 'd_true', 'id'],
      dtype='object')

In [41]:
for column in df.columns:
    print(column)

zero_shot_depression_naive_gpt-3.5-turbo_seed_0
zero_shot_depression_naive_gpt-4_seed_0
zero_shot_depression_mards_gpt-3.5-turbo_seed_0_t=35
zero_shot_depression_mards_gpt-3.5-turbo_seed_0_t=20
zero_shot_depression_mards_gpt-4_seed_0_t=35
zero_shot_depression_mards_gpt-4_seed_0_t=20
zero_shot_depression_phq9_gpt-3.5-turbo_seed_0
zero_shot_depression_phq9_gpt-4_seed_0
d_true
id


### Post-Process FlanT5 predictions

In [12]:
filename = '/home/ameyh/mental-health-comorbitidy-classification/results/few_shot/few_shot_comorbidity_mental_flan_t5_num_examples_ss_2_v1_seed_1.csv'
df_pred = pd.read_csv(filename)
print(df_pred.shape)
df_pred.head(1)

(2872, 13)


,index,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label,few_shot_prompt_comorbidity,exemplar_labels_comorbidity,results_comorbidity_mental_flan_t5
0,0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0,\nBelow are posts and their respective assessm...,"[{'depression': 'yes', 'anxiety': 'no'}, {'dep...",Yes


In [15]:
df_pred['results_comorbidity_mental_flan_t5'].value_counts()

Yes           2568
Behavior        83
Attempt         75
Ideation        69
severe          48
No              19
Indicator        7
Supportive       2
Name: results_comorbidity_mental_flan_t5, dtype: int64

In [18]:
print(df_pred[df_pred['results_comorbidity_mental_flan_t5'] != 'Yes'].iloc[0]['few_shot_prompt_comorbidity'])


Below are posts and their respective assessments based on the criteria for clinical depression and clinical anxiety respectively as defined in the DSM-5.
Format your response as a JSON object {'depression': '', 'anxiety': ''} with values either 'yes' or 'no'.

Post: i am 18, but i am so tired of life that it is unbearable. i have been struggling with depression and anxiety for over 5 years and its just getting worse and worse. i cant bring myself to do anything productive, mainly because my illness and because. life is just too boring for me and i would like to end it. its not like i am getting anywhere since the only thing i do all day is watch anime and game. even if i want to learn, work out or practice guitar, i just cant. if this continues i have no hopes for graduating, going to college, getting decent job or having a stable relationship. so basically i will have really no reason to keep struggling. i am just a big failure atm and i will probably stay like that. lately i have be

In [ ]:
fewshot_template = """
Below are posts and their respective assessments based on the criteria for clinical depression and clinical anxiety respectively as defined in the DSM-5. 

Post: i am 18, but i am so tired of life that it is unbearable. i have been struggling with depression and anxiety for over 5 years and its just getting worse and worse. i cant bring myself to do anything productive, mainly because my illness and because. life is just too boring for me and i would like to end it. its not like i am getting anywhere since the only thing i do all day is watch anime and game. even if i want to learn, work out or practice guitar, i just cant. if this continues i have no hopes for graduating, going to college, getting decent job or having a stable relationship. so basically i will have really no reason to keep struggling. i am just a big failure atm and i will probably stay like that. lately i have been dreaming about suicide and writing my last words way too much. i even remember what i wanted to write there. anyway thank you for reading my vent, i am going to cry myself to sleep now ps please dont send me any jesus loves you etc messages into dms. i am atheist and it will just cause more harm than good
Depression: Yes
Anxiety: No

Post: i dont want to look for a job anymore i dont want to talk to family i dont want to talk to.my friends i dont want to visit my family i want to get up anymore i dont want to eat i dont want to clean i just want to fucking die i want someone to put me out of my misery. but im to lazy to kill myself so ill just waste away for the rest of my life, not something to enjoy but just a matter of fact for me.
Depression: No
Anxiety: Yes

Post: i dont want to be happy, i dont want money, or more friends or anything else. i literally only want to die, thats all i want. truly, genuinely, i want to die. because i know for a fact the only thing i ever wanted in life doesnt exist. im never going to fall in love with a girl like i always dreamed of, ill never get a wholesome romance, my life has shown ill be alone always. as a 24m what i hear most is that im young and i have plenty of time. but if you were on fire would you be ok with someone telling you to wait before jumping into a pool of water? i live in emotional agony and it becomes harder to escape every day. my only relief is death, verily. i only want to die. edit i deeply appreciate the comments on this post, they gave me plenty to think about.
Depression: Yes
Anxiety: Yes

Post: ive been meaning to write this for a long time. im going to try to keep it short. so, i just want to die, seriously. i just cant stand myself, not even anyone, i just feel like im a total letdown to everyone. lately all i see is grey, i try to cheer up but nothing helps, everything right now is grey to me. im aware that a lot of people are struggling with this too, but i still feel so alone. the worst part is that deep down i know that everything is okay, but my mind is flooded with depressive thoughts. i simply dont want to stay alive anymore. lately all i think is about the fact that ive got to suicide before my birthday, simply because i dont want to see anyone congratulating me when i already know that no one likes me. i feel like im a huge piece of shit, i just know that no one wants to be around me, i know that i ruin everything, i know that i cant even write about my problems without feeling useless for not being able to handle them on my own. i just need someone to tell me that everything is going to be okay, even if it isnt going to be okay.
Depression: No
Anxiety: No

Based on the above, assess the content of the following post:
Post: I want to die. 15F Ive always wanted to.that hasnt changed. But now more than ever. Im bad at everything.I suck in school on subjects society cares about, math and science. I cant think at all.so Im stupid. And if Im stupid, then I shouldnt be here. I hate school.I hate learning what I cant understand because of my stupid self. I was born stupid, so Im forever stupid. and I want to die. I dont want to wake up. I really want to die. I probably should since me being here is in no way helping. Im wasting the space here.
"""



In [51]:
def clean(x):
    x = x.replace("Format your response as a JSON object {'anxiety':''} with values either 'yes' or 'no'.\n",'')
    x = x.replace("Format your response as a JSON object {'depression':''} with values either 'yes' or 'no'.\n",'')
    x = x.replace('{','').replace('}','').replace('Assesement: ' ,'').replace('Assessment:' ,'')
    x = x.replace("Format your response as a JSON object 'depression': '', 'anxiety': '' with values either 'yes' or 'no'.\n",'')
    return x.strip()


filename = '/home/ameyh/mental-health-comorbitidy-classification/data/few_shot_prompts/flanT5_comorbidity_ss_only_num_examples_per_label_2.csv'
colname = 'few_shot_prompt_comorbidity'

df = pd.read_csv(filename)
df[colname] = df[colname].apply(lambda x: clean(x))
prompt = df.iloc[0][colname]
print(prompt)

Below are posts and their respective assessments based on the criteria for clinical depression and clinical anxiety respectively as defined in the DSM-5.

Post: before anyone asks, yes, i am diagnosed with depression. ive had it for 7 years but was only diagnosed a few weeks ago. i used to cut to let the pain out. then that stopped helping so it was just to feel something. now i dont even feel anything when i do it. i just do it because i can. and idk why. i feel nuts and im just waiting to get thrown in a hospital.
'depression': 'yes', 'anxiety': 'no'

Post: so, ive been inactive on this account for a while, but im here to just say stuff again like i did in my other two posts or whatever. it seems like ive just started doing it because i got bored. it gives me something to focus on instead of having to think about everything happening at school and home. its also a stress relief for me. whenever im pissed or stressed ill make a scratch or two and it helps a lot. it doesnt seem like th

In [50]:
df.to_csv(filename, index=False)